In [49]:
import pandas as pd
data = pd.read_excel("./Datos aplanados BGV.xlsx")
print(list(data.columns))

['Column1', 'Columna1', 'NUMCAT', 'ACCENUMB', 'Original scientific name', 'World Flora Online name', 'SPAUTHOR', 'SUBTAXA', 'SUBTAUTHOR', 'SUBTAXA_2', 'SUBTAUTHOR_2', 'SUBTAXA_CRF', 'SUBTAUTHOR_CRF', 'FAMILY', 'CROPNAME', 'LOCALNAME', 'ANTHOSNAME', 'Comentario Taxon', 'SILVEMP', 'COLLDATE', 'Comentario Fecha', 'ACQDATE', 'DateLastModified', 'THREATS', 'Comentarios adicionales', 'Motivo de la conservación', 'REMARKS', 'CountryQuaternarySubdivision ', 'Pr', 'ISLE', 'StateProvinceISO', 'CountrySecondarySubdivision ', 'CountryPrimarySubdivision ', 'CountryTertiarySubdivision ', 'ORIGCTY', 'ELEVATION', 'Orientación del terreno', 'HABITATDESCRIPTION', 'VEGETATION', 'SOILTYPE', 'UTM-MGRS', 'Comentarios Localidad', 'LATITUDE', 'LONGITUDE', 'COORDDATUM', 'GEOREFMETH', 'R1', 'R2', 'R3', 'R4', 'recolector1', 'recolector2', 'recolector3', 'recolector4', 'collector', 'COLLMISSID', 'Institución a la que pertenecen*', 'COLLCODE', 'Persona de contacto', 'COLLINSTNAME', 'COLLINSTADDRESS', 'COLLINSTPHON

In [6]:
data.head()

,Column1,Columna1,NUMCAT,ACCENUMB,Original scientific name,World Flora Online name,SPAUTHOR,SUBTAXA,SUBTAUTHOR,SUBTAXA_2,...,Tamaño estimado de la población,Número de individuos recolectados,DONATION,Depósito (indicar restricciones de uso),DUPLICATEMATERIAL,DONORNUMB,Documentación acompañante,DONORCODE,TIRFAA,Permiso
0,10817.0,C,NC058636,1.0,Lobularia maritima,Lobularia maritima,(L.) Desv.,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10818.0,C,NC058637,2.0,Alyssum spinosum,Hormathophylla spinosa,L.,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10819.0,C,NC058638,3.0,Diplotaxis erucoides,Diplotaxis erucoides,(L.) DC.,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10820.0,C,NC058639,4.0,Cakile maritima,Cakile maritima,Scop.,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10821.0,C,NC058640,5.0,Veronica repens,Veronica repens,Clarion ex DC.,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [64]:
# Function to concatenate non-empty values with a space separator
def concatenate_non_empty(row):
    # Filter out empty strings, NaN, and None
    valid_values = [str(val) for val in row if pd.notna(val) and val != '']
    # Join with a single space
    return ' '.join(valid_values)

# Apply the function to concatenate the four columns
data["scientific_name"] = data[['Original scientific name','SUBTAXA', 'SUBTAXA_2']].apply(concatenate_non_empty, axis=1)
data["scientific_name"]

0          Lobularia maritima
1            Alyssum spinosum
2        Diplotaxis erucoides
3             Cakile maritima
4             Veronica repens
                 ...         
10611    Dactylis  glomerata 
10612         Daucus  carota 
10613     Thymus  mastichina 
10614       Lactuca serriola 
10615      Allium  oleraceum 
Name: scientific_name, Length: 10616, dtype: object

In [65]:
germplasm = pd.DataFrame(columns=["uniqid", "accession_number", "national_catalogue_code", "taxon_identifier", "scientific_name", "scientific_name_authorship", "vernacular_name", "taxonomy_reference", "determination_status", "iucn_threat_category"])
germplasm[["accession_number", "national_catalogue_code", "scientific_name", "scientific_name_authorship", "vernacular_name"]] = data[["ACCENUMB", "NUMCAT", "scientific_name" ,"SPAUTHOR", "CROPNAME"]]
germplasm["accession_number"] = germplasm["accession_number"]
germplasm

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category
0,NaN,1.0,NC058636,NaN,Lobularia maritima,(L.) Desv.,Mastuerzo marino,NaN,NaN,NaN
1,NaN,2.0,NC058637,NaN,Alyssum spinosum,L.,NaN,NaN,NaN,NaN
2,NaN,3.0,NC058638,NaN,Diplotaxis erucoides,(L.) DC.,Rabaniza blanca,NaN,NaN,NaN
3,NaN,4.0,NC058639,NaN,Cakile maritima,Scop.,NaN,NaN,NaN,NaN
4,NaN,5.0,NC058640,NaN,Veronica repens,Clarion ex DC.,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
10611,NaN,10603.0,NaN,NaN,Dactylis glomerata,L.,dáctilo; enjilaora,NaN,NaN,NaN
10612,NaN,10604.0,NaN,NaN,Daucus carota,L.,zanahoria silvetre,NaN,NaN,NaN
10613,NaN,10605.0,NaN,NaN,Thymus mastichina,(L.) L.,mejorana; tomillo blanco,NaN,NaN,NaN
10614,NaN,10606.0,NaN,NaN,Lactuca serriola,L.,lechuga silvestre; lechera,NaN,NaN,NaN


In [66]:
df2 = pd.DataFrame({
    "sciname": germplasm["scientific_name"].dropna().drop_duplicates()
})
df2.to_csv("./WFOAPI_input.csv", index = True, index_label="ID")

In [67]:
import subprocess
import os
script_path = "../tnrsapi.sh"
args = [
    "-f", "/home/acb8/Work/FLAIR-GG_models/germplasm_banks/WFOAPI_input.csv",
    "-o", "/home/acb8/Work/FLAIR-GG_models/germplasm_banks/WFOAPI_input_resolved.csv",
    "-m", "resolve",
    "-s", "wfo",
    "-k"
]

try:
    result = subprocess.run(
        [script_path] + args,
        capture_output=True,
        text=True
    )
    print("Script output:")
    print(result.stdout)
    if result.stderr:
        print("Script errors:")
        print(result.stderr)
    if result.returncode == 0:
        print("Script ran successfully.")
        # Check if output CSV exists
        if os.path.exists("/home/acb8/Work/FLAIR-GG_models/germplasm_banks/WFOAPI_input_resolved.csv"):
            print("Output CSV generated successfully.")
        else:
            print("Output CSV not found.")
    else:
        print(f"Script failed with return code {result.returncode}.")
except FileNotFoundError:
    print("Error: tnrsapi.sh not found in current directory.")
except subprocess.CalledProcessError as e:
    print(f"Error running script: {e}")
    print(f"Errors: {e.stderr}")

Script output:
 
Names submitted:
|     ID | sciname                                                       |
| ------ | ------------------------------------------------------------- |
|      0 | Lobularia maritima                                            |
|      1 | Alyssum spinosum                                              |
|      2 | Diplotaxis erucoides                                          |
|      3 | Cakile maritima                                               |
|      4 | Veronica repens                                               |
|      6 | Crambe maritima                                               |
|      7 | Draba alpina                                                  |
|      8 | Pritzelago alpina                                             |
|      9 | Hugueninia tanacetifolia                                      |
|     10 | Glaucium flavum                                               |
|     11 | Arabis alpina cantabrica                               

In [68]:
import numpy as np
dfWFO = pd.read_csv("/home/acb8/Work/FLAIR-GG_models/germplasm_banks/WFOAPI_input_resolved.csv")
dfWFO["scientific_name"] = np.where(
    dfWFO["Accepted_name"].notna() & (dfWFO["Accepted_name"] != ""),
    dfWFO["Accepted_name"],
    dfWFO["Name_matched"]
)
dfWFO["taxon_identifier"] = np.where(
    dfWFO["Accepted_name_url"].notna() & (dfWFO["Accepted_name_url"] != ""),
    dfWFO["Accepted_name_url"],
    dfWFO["Name_matched_url"]
)

dfWFO["scientific_name_authorship"] = np.where(
    dfWFO["Accepted_name_author"].notna() & (dfWFO["Accepted_name_author"] != ""),
    dfWFO["Accepted_name_author"],
    dfWFO["Canonical_author"]
)
dfWFO = dfWFO[["Name_submitted", "Author_submitted", "scientific_name", "taxon_identifier", "scientific_name_authorship"]]

dfWFO

,Name_submitted,Author_submitted,scientific_name,taxon_identifier,scientific_name_authorship
0,Lobularia maritima,NaN,Lobularia maritima,https://www.worldfloraonline.org/taxon/wfo-000...,(L.) Desv.
1,Alyssum spinosum,NaN,Hormathophylla spinosa,https://www.worldfloraonline.org/taxon/wfo-000...,(L.) P.Küpfer
2,Diplotaxis erucoides,NaN,Diplotaxis erucoides,https://www.worldfloraonline.org/taxon/wfo-000...,(L.) DC.
3,Cakile maritima,NaN,Cakile maritima,https://www.worldfloraonline.org/taxon/wfo-000...,Scop.
4,Veronica repens,NaN,Veronica repens,https://www.worldfloraonline.org/taxon/wfo-000...,Clarion ex DC.
...,...,...,...,...,...
4129,Dactylis glomerata,NaN,Dactylis glomerata,https://www.worldfloraonline.org/taxon/wfo-000...,L.
4130,Daucus carota,NaN,Daucus carota,https://www.worldfloraonline.org/taxon/wfo-000...,L.
4131,Thymus mastichina,NaN,Thymus mastichina,https://www.worldfloraonline.org/taxon/wfo-000...,L.
4132,Lactuca serriola,NaN,Lactuca serriola,https://www.worldfloraonline.org/taxon/wfo-000...,L.


In [69]:
import unicodedata
# Function to remove diacritics
def remove_diacritics(text):
    if pd.isna(text):
        return text
    # Normalize to decomposed form (separate character and diacritic)
    normalized = unicodedata.normalize('NFD', str(text))
    # Remove non-ASCII characters (diacritics)
    return ''.join(c for c in normalized if unicodedata.category(c) != 'Mn').strip()
    
germplasm["temp_sci_name"] = germplasm["scientific_name"].apply(remove_diacritics)
#replace commas with space
germplasm["temp_sci_name"] = germplasm["temp_sci_name"].str.replace(',', ' ', regex=False)

In [70]:
merged_df = pd.merge(germplasm, dfWFO, left_on = "temp_sci_name", right_on = "Name_submitted", how = "inner", suffixes = ("_germplasm", "_wfo"))
merged_df

,uniqid,accession_number,national_catalogue_code,taxon_identifier_germplasm,scientific_name_germplasm,scientific_name_authorship_germplasm,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category,temp_sci_name,Name_submitted,Author_submitted,scientific_name_wfo,taxon_identifier_wfo,scientific_name_authorship_wfo
0,NaN,1.0,NC058636,NaN,Lobularia maritima,(L.) Desv.,Mastuerzo marino,NaN,NaN,NaN,Lobularia maritima,Lobularia maritima,NaN,Lobularia maritima,https://www.worldfloraonline.org/taxon/wfo-000...,(L.) Desv.
1,NaN,2.0,NC058637,NaN,Alyssum spinosum,L.,NaN,NaN,NaN,NaN,Alyssum spinosum,Alyssum spinosum,NaN,Hormathophylla spinosa,https://www.worldfloraonline.org/taxon/wfo-000...,(L.) P.Küpfer
2,NaN,3.0,NC058638,NaN,Diplotaxis erucoides,(L.) DC.,Rabaniza blanca,NaN,NaN,NaN,Diplotaxis erucoides,Diplotaxis erucoides,NaN,Diplotaxis erucoides,https://www.worldfloraonline.org/taxon/wfo-000...,(L.) DC.
3,NaN,4.0,NC058639,NaN,Cakile maritima,Scop.,NaN,NaN,NaN,NaN,Cakile maritima,Cakile maritima,NaN,Cakile maritima,https://www.worldfloraonline.org/taxon/wfo-000...,Scop.
4,NaN,5.0,NC058640,NaN,Veronica repens,Clarion ex DC.,NaN,NaN,NaN,NaN,Veronica repens,Veronica repens,NaN,Veronica repens,https://www.worldfloraonline.org/taxon/wfo-000...,Clarion ex DC.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10410,NaN,10593.0,NaN,NaN,Ornithopus perpusillus,L.,NaN,NaN,NaN,NaN,Ornithopus perpusillus,Ornithopus perpusillus,NaN,Ornithopus perpusillus,https://www.worldfloraonline.org/taxon/wfo-000...,L.
10411,NaN,10594.0,NaN,NaN,Lotus corniculatus,L.,NaN,NaN,NaN,NaN,Lotus corniculatus,Lotus corniculatus,NaN,Lotus corniculatus,https://www.worldfloraonline.org/taxon/wfo-000...,L.
10412,NaN,10595.0,NaN,NaN,Trifolium dubium,Sibth.,NaN,NaN,NaN,NaN,Trifolium dubium,Trifolium dubium,NaN,Trifolium dubium,https://www.worldfloraonline.org/taxon/wfo-000...,Sibth.
10413,NaN,10596.0,NaN,NaN,Lavandula pedunculata,(Mill.) Cav.,NaN,NaN,NaN,NaN,Lavandula pedunculata,Lavandula pedunculata,NaN,Lavandula pedunculata,https://www.worldfloraonline.org/taxon/wfo-000...,(Mill.) Cav.


In [71]:
merged_df = merged_df[["uniqid", "accession_number", "national_catalogue_code", "taxon_identifier_wfo", "scientific_name_wfo","scientific_name_authorship_wfo","vernacular_name", "taxonomy_reference", "determination_status", "iucn_threat_category"]]
merged_df["taxonomy_reference"] = "WFO (2025): World Flora Online. Published on the Internet; http://www.worldfloraonline.org"
merged_df.rename(columns={"taxon_identifier_wfo" : "taxon_identifier", "scientific_name_wfo" : "scientific_name", "scientific_name_authorship_wfo" : "scientific_name_authorship"}, inplace = True)
merged_df

/tmp/ipykernel_4904/411484739.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df["taxonomy_reference"] = "WFO (2025): World Flora Online. Published on the Internet; http://www.worldfloraonline.org"
/tmp/ipykernel_4904/411484739.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df.rename(columns={"taxon_identifier_wfo" : "taxon_identifier", "scientific_name_wfo" : "scientific_name", "scientific_name_authorship_wfo" : "scientific_name_authorship"}, inplace = True)


,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category
0,NaN,1.0,NC058636,https://www.worldfloraonline.org/taxon/wfo-000...,Lobularia maritima,(L.) Desv.,Mastuerzo marino,WFO (2025): World Flora Online. Published on t...,NaN,NaN
1,NaN,2.0,NC058637,https://www.worldfloraonline.org/taxon/wfo-000...,Hormathophylla spinosa,(L.) P.Küpfer,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
2,NaN,3.0,NC058638,https://www.worldfloraonline.org/taxon/wfo-000...,Diplotaxis erucoides,(L.) DC.,Rabaniza blanca,WFO (2025): World Flora Online. Published on t...,NaN,NaN
3,NaN,4.0,NC058639,https://www.worldfloraonline.org/taxon/wfo-000...,Cakile maritima,Scop.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
4,NaN,5.0,NC058640,https://www.worldfloraonline.org/taxon/wfo-000...,Veronica repens,Clarion ex DC.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
10410,NaN,10593.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Ornithopus perpusillus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
10411,NaN,10594.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Lotus corniculatus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
10412,NaN,10595.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Trifolium dubium,Sibth.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
10413,NaN,10596.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Lavandula pedunculata,(Mill.) Cav.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN


In [72]:
germplasm["taxonomy_reference"] = "Lista patrón de las especies silvestres presentes en España (MITECO, 2024)"
germplasm

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category,temp_sci_name
0,NaN,1.0,NC058636,NaN,Lobularia maritima,(L.) Desv.,Mastuerzo marino,Lista patrón de las especies silvestres presen...,NaN,NaN,Lobularia maritima
1,NaN,2.0,NC058637,NaN,Alyssum spinosum,L.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN,Alyssum spinosum
2,NaN,3.0,NC058638,NaN,Diplotaxis erucoides,(L.) DC.,Rabaniza blanca,Lista patrón de las especies silvestres presen...,NaN,NaN,Diplotaxis erucoides
3,NaN,4.0,NC058639,NaN,Cakile maritima,Scop.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN,Cakile maritima
4,NaN,5.0,NC058640,NaN,Veronica repens,Clarion ex DC.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN,Veronica repens
...,...,...,...,...,...,...,...,...,...,...,...
10611,NaN,10603.0,NaN,NaN,Dactylis glomerata,L.,dáctilo; enjilaora,Lista patrón de las especies silvestres presen...,NaN,NaN,Dactylis glomerata
10612,NaN,10604.0,NaN,NaN,Daucus carota,L.,zanahoria silvetre,Lista patrón de las especies silvestres presen...,NaN,NaN,Daucus carota
10613,NaN,10605.0,NaN,NaN,Thymus mastichina,(L.) L.,mejorana; tomillo blanco,Lista patrón de las especies silvestres presen...,NaN,NaN,Thymus mastichina
10614,NaN,10606.0,NaN,NaN,Lactuca serriola,L.,lechuga silvestre; lechera,Lista patrón de las especies silvestres presen...,NaN,NaN,Lactuca serriola


In [73]:
merged_df

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category
0,NaN,1.0,NC058636,https://www.worldfloraonline.org/taxon/wfo-000...,Lobularia maritima,(L.) Desv.,Mastuerzo marino,WFO (2025): World Flora Online. Published on t...,NaN,NaN
1,NaN,2.0,NC058637,https://www.worldfloraonline.org/taxon/wfo-000...,Hormathophylla spinosa,(L.) P.Küpfer,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
2,NaN,3.0,NC058638,https://www.worldfloraonline.org/taxon/wfo-000...,Diplotaxis erucoides,(L.) DC.,Rabaniza blanca,WFO (2025): World Flora Online. Published on t...,NaN,NaN
3,NaN,4.0,NC058639,https://www.worldfloraonline.org/taxon/wfo-000...,Cakile maritima,Scop.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
4,NaN,5.0,NC058640,https://www.worldfloraonline.org/taxon/wfo-000...,Veronica repens,Clarion ex DC.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
10410,NaN,10593.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Ornithopus perpusillus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
10411,NaN,10594.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Lotus corniculatus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
10412,NaN,10595.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Trifolium dubium,Sibth.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
10413,NaN,10596.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Lavandula pedunculata,(Mill.) Cav.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN


In [74]:
germplasm = germplasm.drop(columns=["temp_sci_name"])
germplasm2 = pd.concat([germplasm, merged_df], ignore_index = True)
germplasm2

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category
0,NaN,1.0,NC058636,NaN,Lobularia maritima,(L.) Desv.,Mastuerzo marino,Lista patrón de las especies silvestres presen...,NaN,NaN
1,NaN,2.0,NC058637,NaN,Alyssum spinosum,L.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN
2,NaN,3.0,NC058638,NaN,Diplotaxis erucoides,(L.) DC.,Rabaniza blanca,Lista patrón de las especies silvestres presen...,NaN,NaN
3,NaN,4.0,NC058639,NaN,Cakile maritima,Scop.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN
4,NaN,5.0,NC058640,NaN,Veronica repens,Clarion ex DC.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
21026,NaN,10593.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Ornithopus perpusillus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
21027,NaN,10594.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Lotus corniculatus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
21028,NaN,10595.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Trifolium dubium,Sibth.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
21029,NaN,10596.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Lavandula pedunculata,(Mill.) Cav.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN


Now I'm going to fill the iucn_threat_category column. To do so, I'm going to use the national IUCN red list for Spain, created by SEBICOP

In [75]:
iucn = pd.read_excel("lista roja sebicop wfo.xlsx", sheet_name = 1)
iucn = iucn.drop(columns = "WFO name")
iucn.head()

,Sebicop Original name,Category
0,Anthemis funkii,EX
1,Carduncellus matritensis,EX *
2,Pharbitis preauxii,EX
3,Normania nava,EX
4,Aeonium mascaense,EW


In [76]:
df2 = pd.DataFrame({
    "sciname": iucn["Sebicop Original name"].dropna().drop_duplicates()
})
df2.to_csv("./WFOAPI_input.csv", index = True, index_label="ID")

In [77]:
import subprocess
import os
script_path = "../tnrsapi.sh"
args = [
    "-f", "/home/acb8/Work/FLAIR-GG_models/germplasm_banks/WFOAPI_input.csv",
    "-o", "/home/acb8/Work/FLAIR-GG_models/germplasm_banks/WFOAPI_input_resolved.csv",
    "-m", "resolve",
    "-s", "wfo",
    "-k"
]

try:
    result = subprocess.run(
        [script_path] + args,
        capture_output=True,
        text=True
    )
    print("Script output:")
    print(result.stdout)
    if result.stderr:
        print("Script errors:")
        print(result.stderr)
    if result.returncode == 0:
        print("Script ran successfully.")
        # Check if output CSV exists
        if os.path.exists("/home/acb8/Work/FLAIR-GG_models/germplasm_banks/WFOAPI_input_resolved.csv"):
            print("Output CSV generated successfully.")
        else:
            print("Output CSV not found.")
    else:
        print(f"Script failed with return code {result.returncode}.")
except FileNotFoundError:
    print("Error: tnrsapi.sh not found in current directory.")
except subprocess.CalledProcessError as e:
    print(f"Error running script: {e}")
    print(f"Errors: {e.stderr}")

Script output:
 
Names submitted:
|    ID | sciname                                                      |
| ----- | ------------------------------------------------------------ |
|     0 | Anthemis funkii                                              |
|     1 | Carduncellus matritensis                                     |
|     2 | Pharbitis preauxii                                           |
|     3 | Normania nava                                                |
|     4 | Aeonium mascaense                                            |
|     5 | Linaria polygalifolia subsp. lamarckii                       |
|     6 | Lysimachia minoricensis                                      |
|     7 | Adoxa moschatellina                                          |
|     8 | Sagittaria sagittifolia                                      |
|     9 | Pulicaria undulata subsp. undulata                           |
|    10 | Elizaldia calycina subsp. multicolor                         |
|    11 | Aurinia

In [78]:
import numpy as np
dfWFO = pd.read_csv("/home/acb8/Work/FLAIR-GG_models/germplasm_banks/WFOAPI_input_resolved.csv")
dfWFO["scientific_name"] = np.where(
    dfWFO["Accepted_name"].notna() & (dfWFO["Accepted_name"] != ""),
    dfWFO["Accepted_name"],
    dfWFO["Name_matched"]
)
dfWFO["taxon_identifier"] = np.where(
    dfWFO["Accepted_name_url"].notna() & (dfWFO["Accepted_name_url"] != ""),
    dfWFO["Accepted_name_url"],
    dfWFO["Name_matched_url"]
)

dfWFO = dfWFO[["Name_submitted","scientific_name", "taxon_identifier"]]

dfWFO

,Name_submitted,scientific_name,taxon_identifier
0,Anthemis funkii,Anthemis funkii,https://www.worldfloraonline.org/taxon/wfo-100...
1,Carduncellus matritensis,Carduncellus matritensis,https://www.worldfloraonline.org/taxon/wfo-000...
2,Pharbitis preauxii,Ipomoea imperati,https://www.worldfloraonline.org/taxon/wfo-000...
3,Normania nava,Solanum nava,https://www.worldfloraonline.org/taxon/wfo-000...
4,Aeonium mascaense,Aeonium mascaense,https://www.worldfloraonline.org/taxon/wfo-000...
...,...,...,...
1630,Galium geminiflorum,Galium geminiflorum,https://www.worldfloraonline.org/taxon/wfo-000...
1631,Scrophularia oxyrrhyncha,Scrophularia oxyrhyncha,https://www.worldfloraonline.org/taxon/wfo-000...
1632,Verbascum prunellii,Verbascum prunellii,https://www.worldfloraonline.org/taxon/wfo-000...
1633,Thymelaea lanuginosa,Thymelaea lanuginosa,https://www.worldfloraonline.org/taxon/wfo-000...


In [79]:
import unicodedata
# Function to remove diacritics
def remove_diacritics(text):
    if pd.isna(text):
        return text
    # Normalize to decomposed form (separate character and diacritic)
    normalized = unicodedata.normalize('NFD', str(text))
    # Remove non-ASCII characters (diacritics)
    return ''.join(c for c in normalized if unicodedata.category(c) != 'Mn').strip()
    
iucn["temp_sci_name"] = iucn["Sebicop Original name"].apply(remove_diacritics)
#replace commas with space
iucn["temp_sci_name"] = iucn["temp_sci_name"].str.replace(',', ' ', regex=False)

In [80]:
merged_df = pd.merge(iucn, dfWFO, left_on = "temp_sci_name", right_on = "Name_submitted", how = "inner", suffixes = ("_germplasm", "_wfo"))
merged_df

,Sebicop Original name,Category,temp_sci_name,Name_submitted,scientific_name,taxon_identifier
0,Anthemis funkii,EX,Anthemis funkii,Anthemis funkii,Anthemis funkii,https://www.worldfloraonline.org/taxon/wfo-100...
1,Carduncellus matritensis,EX *,Carduncellus matritensis,Carduncellus matritensis,Carduncellus matritensis,https://www.worldfloraonline.org/taxon/wfo-000...
2,Pharbitis preauxii,EX,Pharbitis preauxii,Pharbitis preauxii,Ipomoea imperati,https://www.worldfloraonline.org/taxon/wfo-000...
3,Normania nava,EX,Normania nava,Normania nava,Solanum nava,https://www.worldfloraonline.org/taxon/wfo-000...
4,Aeonium mascaense,EW,Aeonium mascaense,Aeonium mascaense,Aeonium mascaense,https://www.worldfloraonline.org/taxon/wfo-000...
...,...,...,...,...,...,...
1630,Galium geminiflorum,DD,Galium geminiflorum,Galium geminiflorum,Galium geminiflorum,https://www.worldfloraonline.org/taxon/wfo-000...
1631,Scrophularia oxyrrhyncha,DD,Scrophularia oxyrrhyncha,Scrophularia oxyrrhyncha,Scrophularia oxyrhyncha,https://www.worldfloraonline.org/taxon/wfo-000...
1632,Verbascum prunellii,DD,Verbascum prunellii,Verbascum prunellii,Verbascum prunellii,https://www.worldfloraonline.org/taxon/wfo-000...
1633,Thymelaea lanuginosa,DD,Thymelaea lanuginosa,Thymelaea lanuginosa,Thymelaea lanuginosa,https://www.worldfloraonline.org/taxon/wfo-000...


In [128]:
threat = merged_df[["scientific_name", "taxon_identifier", "Category"]]
threat["Category"] = threat["Category"].str.replace(" *", "", regex = False)
threat.rename(columns = {"taxon_identifier" : "taxon_identifier_iucn"}, inplace = True)
threat["taxon_identifier_iucn"] = threat["taxon_identifier_iucn"].fillna("[No match]")
threat

/tmp/ipykernel_4904/1447459377.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  threat["Category"] = threat["Category"].str.replace(" *", "", regex = False)
/tmp/ipykernel_4904/1447459377.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  threat.rename(columns = {"taxon_identifier" : "taxon_identifier_iucn"}, inplace = True)
/tmp/ipykernel_4904/1447459377.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas

,scientific_name,taxon_identifier_iucn,Category
0,Anthemis funkii,https://www.worldfloraonline.org/taxon/wfo-100...,EX
1,Carduncellus matritensis,https://www.worldfloraonline.org/taxon/wfo-000...,EX
2,Ipomoea imperati,https://www.worldfloraonline.org/taxon/wfo-000...,EX
3,Solanum nava,https://www.worldfloraonline.org/taxon/wfo-000...,EX
4,Aeonium mascaense,https://www.worldfloraonline.org/taxon/wfo-000...,EW
...,...,...,...
1630,Galium geminiflorum,https://www.worldfloraonline.org/taxon/wfo-000...,DD
1631,Scrophularia oxyrhyncha,https://www.worldfloraonline.org/taxon/wfo-000...,DD
1632,Verbascum prunellii,https://www.worldfloraonline.org/taxon/wfo-000...,DD
1633,Thymelaea lanuginosa,https://www.worldfloraonline.org/taxon/wfo-000...,DD


In [129]:
threat[threat["taxon_identifier_iucn"].isna()]

,scientific_name,taxon_identifier_iucn,Category


In [118]:
germplasm2

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category
0,NaN,1.0,NC058636,NaN,Lobularia maritima,(L.) Desv.,Mastuerzo marino,Lista patrón de las especies silvestres presen...,NaN,NaN
1,NaN,2.0,NC058637,NaN,Alyssum spinosum,L.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN
2,NaN,3.0,NC058638,NaN,Diplotaxis erucoides,(L.) DC.,Rabaniza blanca,Lista patrón de las especies silvestres presen...,NaN,NaN
3,NaN,4.0,NC058639,NaN,Cakile maritima,Scop.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN
4,NaN,5.0,NC058640,NaN,Veronica repens,Clarion ex DC.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
21026,NaN,10593.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Ornithopus perpusillus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
21027,NaN,10594.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Lotus corniculatus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
21028,NaN,10595.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Trifolium dubium,Sibth.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN
21029,NaN,10596.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Lavandula pedunculata,(Mill.) Cav.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN


In [137]:
merged_iucn = pd.merge(germplasm2, threat, left_on = "taxon_identifier", right_on = "taxon_identifier_iucn", how = "left", suffixes = ("_germplasm", "_iucn"))
merged_iucn

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name_germplasm,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category,scientific_name_iucn,taxon_identifier_iucn,Category
0,NaN,1.0,NC058636,NaN,Lobularia maritima,(L.) Desv.,Mastuerzo marino,Lista patrón de las especies silvestres presen...,NaN,NaN,NaN,NaN,NaN
1,NaN,2.0,NC058637,NaN,Alyssum spinosum,L.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN,NaN,NaN,NaN
2,NaN,3.0,NC058638,NaN,Diplotaxis erucoides,(L.) DC.,Rabaniza blanca,Lista patrón de las especies silvestres presen...,NaN,NaN,NaN,NaN,NaN
3,NaN,4.0,NC058639,NaN,Cakile maritima,Scop.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN,NaN,NaN,NaN
4,NaN,5.0,NC058640,NaN,Veronica repens,Clarion ex DC.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21081,NaN,10593.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Ornithopus perpusillus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN,NaN,NaN,NaN
21082,NaN,10594.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Lotus corniculatus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN,NaN,NaN,NaN
21083,NaN,10595.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Trifolium dubium,Sibth.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN,NaN,NaN,NaN
21084,NaN,10596.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Lavandula pedunculata,(Mill.) Cav.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN,NaN,NaN,NaN


In [140]:
merged_iucn["iucn_threat_category"] = "http://rs.gbif.org/vocabulary/iucn/threat_status/" + merged_iucn["Category"]
merged_iucn

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name_germplasm,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category,scientific_name_iucn,taxon_identifier_iucn,Category
0,NaN,1.0,NC058636,NaN,Lobularia maritima,(L.) Desv.,Mastuerzo marino,Lista patrón de las especies silvestres presen...,NaN,NaN,NaN,NaN,NaN
1,NaN,2.0,NC058637,NaN,Alyssum spinosum,L.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN,NaN,NaN,NaN
2,NaN,3.0,NC058638,NaN,Diplotaxis erucoides,(L.) DC.,Rabaniza blanca,Lista patrón de las especies silvestres presen...,NaN,NaN,NaN,NaN,NaN
3,NaN,4.0,NC058639,NaN,Cakile maritima,Scop.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN,NaN,NaN,NaN
4,NaN,5.0,NC058640,NaN,Veronica repens,Clarion ex DC.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21081,NaN,10593.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Ornithopus perpusillus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN,NaN,NaN,NaN
21082,NaN,10594.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Lotus corniculatus,L.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN,NaN,NaN,NaN
21083,NaN,10595.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Trifolium dubium,Sibth.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN,NaN,NaN,NaN
21084,NaN,10596.0,NaN,https://www.worldfloraonline.org/taxon/wfo-000...,Lavandula pedunculata,(Mill.) Cav.,NaN,WFO (2025): World Flora Online. Published on t...,NaN,NaN,NaN,NaN,NaN


In [141]:
output = merged_iucn[["uniqid", "accession_number", "national_catalogue_code", "taxon_identifier", "scientific_name_germplasm", "scientific_name_authorship", "vernacular_name", "taxonomy_reference", "determination_status", "iucn_threat_category"]]
output = output.rename(columns = {"scientific_name_germplasm" : "scientific_name"})

In [143]:
from datetime import datetime
import time
lista = []
for i in range(1, len(output["uniqid"])+1):
    now = datetime.now()
    now = now.strftime('%Y%m%d%H%M%S%f')
    lista.append(now)
    time.sleep(0.001)
output["uniqid"] = lista
output.head()

,uniqid,accession_number,national_catalogue_code,taxon_identifier,scientific_name,scientific_name_authorship,vernacular_name,taxonomy_reference,determination_status,iucn_threat_category
0,20250611154235463176,1.0,NC058636,NaN,Lobularia maritima,(L.) Desv.,Mastuerzo marino,Lista patrón de las especies silvestres presen...,NaN,NaN
1,20250611154235466598,2.0,NC058637,NaN,Alyssum spinosum,L.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN
2,20250611154235468214,3.0,NC058638,NaN,Diplotaxis erucoides,(L.) DC.,Rabaniza blanca,Lista patrón de las especies silvestres presen...,NaN,NaN
3,20250611154235469275,4.0,NC058639,NaN,Cakile maritima,Scop.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN
4,20250611154235470664,5.0,NC058640,NaN,Veronica repens,Clarion ex DC.,NaN,Lista patrón de las especies silvestres presen...,NaN,NaN


In [145]:
output.to_csv("germplasm.csv")

# Location

In [144]:
location = pd.DataFrame(columns=["uniqid", "accession_number", "national_catalogue_code", "acquisition_time", "collecting_time", "soil_type", "latitude", "longitude", "coordinate_uncertainty", "geodetic_datum", "maximum_elevation", "minimum_elevation", "terrain_inclination", "country_name", "country_2letter_code", "first_admin_subdivision", "second_admin_subdivision", "third_admin_subdivision", "fourth_admin_subdivision", "locality_description"])
location

,uniqid,accession_number,national_catalogue_code,acquisition_time,collecting_time,soil_type,latitude,longitude,coordinate_uncertainty,geodetic_datum,maximum_elevation,minimum_elevation,terrain_inclination,country_name,country_2letter_code,first_admin_subdivision,second_admin_subdivision,third_admin_subdivision,fourth_admin_subdivision,locality_description


In [149]:
print(list(data.columns))

['Column1', 'Columna1', 'NUMCAT', 'ACCENUMB', 'Original scientific name', 'World Flora Online name', 'SPAUTHOR', 'SUBTAXA', 'SUBTAUTHOR', 'SUBTAXA_2', 'SUBTAUTHOR_2', 'SUBTAXA_CRF', 'SUBTAUTHOR_CRF', 'FAMILY', 'CROPNAME', 'LOCALNAME', 'ANTHOSNAME', 'Comentario Taxon', 'SILVEMP', 'COLLDATE', 'Comentario Fecha', 'ACQDATE', 'DateLastModified', 'THREATS', 'Comentarios adicionales', 'Motivo de la conservación', 'REMARKS', 'CountryQuaternarySubdivision ', 'Pr', 'ISLE', 'StateProvinceISO', 'CountrySecondarySubdivision ', 'CountryPrimarySubdivision ', 'CountryTertiarySubdivision ', 'ORIGCTY', 'ELEVATION', 'Orientación del terreno', 'HABITATDESCRIPTION', 'VEGETATION', 'SOILTYPE', 'UTM-MGRS', 'Comentarios Localidad', 'LATITUDE', 'LONGITUDE', 'COORDDATUM', 'GEOREFMETH', 'R1', 'R2', 'R3', 'R4', 'recolector1', 'recolector2', 'recolector3', 'recolector4', 'collector', 'COLLMISSID', 'Institución a la que pertenecen*', 'COLLCODE', 'Persona de contacto', 'COLLINSTNAME', 'COLLINSTADDRESS', 'COLLINSTPHON

In [155]:
location[["accession_number", "national_catalogue_code", "acquisition_time", "collecting_time", "latitude", "longitude", "geodetic_datum", "maximum_elevation", "country_name", "first_admin_subdivision", "second_admin_subdivision", "third_admin_subdivision", "fourth_admin_subdivision"]] = data[["ACCENUMB", "NUMCAT", "ACQDATE", "COLLDATE", "LATITUDE", "LONGITUDE", "COORDDATUM", "ELEVATION", "ORIGCTY", "CountryPrimarySubdivision ", "CountrySecondarySubdivision ", "CountryTertiarySubdivision ", "CountryQuaternarySubdivision "]]
location["minimum_elevation"] = location["maximum_elevation"]
location

,uniqid,accession_number,national_catalogue_code,acquisition_time,collecting_time,soil_type,latitude,longitude,coordinate_uncertainty,geodetic_datum,maximum_elevation,minimum_elevation,terrain_inclination,country_name,country_2letter_code,first_admin_subdivision,second_admin_subdivision,third_admin_subdivision,fourth_admin_subdivision,locality_description
0,NaN,1.0,NC058636,1960.0,1960----,NaN,3934--N,00239--E,NaN,NaN,15.0,15.0,NaN,ESP,NaN,Baleares,Baleares,Palma de Mallorca,Palma de Mallorca,NaN
1,NaN,2.0,NC058637,1960.0,1960----,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ESP,NaN,Andalucia,Granada,NaN,"Sierra Nevada,Veleta",NaN
2,NaN,3.0,NC058638,1960.0,1960----,NaN,3934--N,00239--E,NaN,NaN,15.0,15.0,NaN,ESP,NaN,Baleares,Baleares,Palma de Mallorca,Palma de Mallorca,NaN
3,NaN,4.0,NC058639,1960.0,1960----,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J.B. Sacavem,NaN
4,NaN,5.0,NC058640,1960.0,1960----,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ESP,NaN,Andalucia,Granada,NaN,"Sierra Nevada,Veleta",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10611,NaN,10603.0,NaN,NaN,2021-07-28 00:00:00,NaN,41061302,-3535046,NaN,WGS84,NaN,NaN,NaN,ESP,NaN,"Madrid, Comunidad de",MADRID,Montejo de la Sierra,"Finca privada, Reserva de la Biosfera Sierra d...",NaN
10612,NaN,10604.0,NaN,NaN,2021-07-29 00:00:00,NaN,41042947,-3566698,NaN,WGS84,NaN,NaN,NaN,ESP,NaN,"Madrid, Comunidad de",MADRID,Madarcos,"Finca privada, Reserva de la Biosfera Sierra d...",NaN
10613,NaN,10605.0,NaN,NaN,2021-07-29 00:00:00,NaN,410431,-3565898,NaN,WGS84,NaN,NaN,NaN,ESP,NaN,"Madrid, Comunidad de",MADRID,Madarcos,"Finca privada, Reserva de la Biosfera Sierra d...",NaN
10614,NaN,10606.0,NaN,NaN,2021-08-19 00:00:00,NaN,4106129,-3535056,NaN,WGS84,NaN,NaN,NaN,ESP,NaN,"Madrid, Comunidad de",MADRID,Montejo de la Sierra,"Finca privada, Reserva de la Biosfera Sierra d...",NaN


In [167]:
#These mappings have been created by Grok
mapeo_suelos = {
    'sobre pizarras verdes del Cámbrico': 'envo:00002007',  # metamorphic rock soil
    'Complejo basal': 'envo:00002034',  # basaltic soil
    'Lavas volcánicas': 'envo:00002030',  # volcanic soil
    'Coladas de lava': 'envo:00002030',  # volcanic soil
    'Malpaís': 'envo:00002030',  # volcanic soil
    'volcánico, lavas': 'envo:00002030',  # volcanic soil
    'Granitos': 'envo:00002013',  # granitic soil
    'arenoso': 'envo:00002031',  # sandy soil
    'Calcario': 'envo:00002037',  # calcareous soil
    'Arenoso': 'envo:00002031',  # sandy soil
    'Devónico y Carbonífero': 'envo:00002007',  # sedimentary rock soil (con nota de ambigüedad)
    'Calizo': 'envo:00002037',  # calcareous soil
    'Conglomerado': 'envo:00002007',  # sedimentary rock soil
    'Granito': 'envo:00002013',  # granitic soil
    'Esquisto': 'envo:00002007',  # metamorphic rock soil
    'Pedregoso, cantos rodados': 'envo:00002040',  # stony soil
    'Arcilloso': 'envo:00002033',  # clay soil
    'Ácido': 'envo:00001999',  # acidic soil
    'Calizo-dolomítico': 'envo:00002037',  # calcareous soil
    'Sustrato pedregoso calcáreo': 'envo:00002037',  # calcareous soil
    'arenisca': 'envo:00002031',  # sandy soil
    'Pedregoso calcáreo': 'envo:00002037',  # calcareous soil
    'Silíceo': 'envo:00002007',  # siliceous soil
    'Pedregoso-calcáreo': 'envo:00002037',  # calcareous soil
    'pedregoso calcáreo': 'envo:00002037',  # calcareous soil
    'pedregoso calizo': 'envo:00002037',  # calcareous soil
    'Calizo-Margoso': 'envo:00002037',  # calcareous soil
    'Margoso': 'envo:00002037',  # calcareous soil
    'Margoso-yesífero': 'envo:00002037',  # calcareous soil
    'Yesoso': 'envo:00002007',  # sedimentary rock soil
    'sustrato margosos-calizo': 'envo:00002037',  # calcareous soil
    'Roca Calcárea': 'envo:00002037',  # calcareous soil
    'Humus': 'envo:00002036',  # humus soil
    'Calcáreo rocoso': 'envo:00002037',  # calcareous soil
    'Grietas de roca calcárea': 'envo:00002037',  # calcareous soil
    'Argiloso': 'envo:00002033'  # clay soil
}

def mapear_suelos(df, columna):
    """
    Reemplaza los valores en la columna especificada del DataFrame con los términos ENVO mapeados.
    
    Parámetros:
    df (pandas.DataFrame): DataFrame que contiene la columna con tipos de suelo.
    columna (str): Nombre de la columna a mapear.
    
    Retorna:
    pandas.DataFrame: DataFrame con la columna mapeada.
    """
    # Crear una copia del DataFrame para evitar modificar el original
    df_mapeado = df.copy()
    
    # Reemplazar los valores usando el diccionario de mapeo
    df_mapeado[columna] = df_mapeado[columna].map(mapeo_suelos)
    
    # Opcional: Manejar valores no mapeados (NaN) si hay suelos no incluidos en el diccionario
    if df_mapeado[columna].isna().any():
        print("Advertencia: Algunos valores no fueron mapeados. Revisa los valores originales:")
        print(df[df_mapeado[columna].isna()][columna].unique())
    
    return df_mapeado


testdf = mapear_suelos(data, "SOILTYPE")
testdf

Advertencia: Algunos valores no fueron mapeados. Revisa los valores originales:
[nan]


,Column1,Columna1,NUMCAT,ACCENUMB,Original scientific name,World Flora Online name,SPAUTHOR,SUBTAXA,SUBTAUTHOR,SUBTAXA_2,...,Número de individuos recolectados,DONATION,Depósito (indicar restricciones de uso),DUPLICATEMATERIAL,DONORNUMB,Documentación acompañante,DONORCODE,TIRFAA,Permiso,scientific_name
0,10817.0,C,NC058636,1.0,Lobularia maritima,Lobularia maritima,(L.) Desv.,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Lobularia maritima
1,10818.0,C,NC058637,2.0,Alyssum spinosum,Hormathophylla spinosa,L.,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Alyssum spinosum
2,10819.0,C,NC058638,3.0,Diplotaxis erucoides,Diplotaxis erucoides,(L.) DC.,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Diplotaxis erucoides
3,10820.0,C,NC058639,4.0,Cakile maritima,Cakile maritima,Scop.,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cakile maritima
4,10821.0,C,NC058640,5.0,Veronica repens,Veronica repens,Clarion ex DC.,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Veronica repens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10611,23596.0,E,NaN,10603.0,Dactylis glomerata,Dactylis glomerata,L.,NaN,NaN,NaN,...,NaN,NaN,Si,NaN,NaN,NaN,NaN,TIRFAA,NaN,Dactylis glomerata
10612,23597.0,E,NaN,10604.0,Daucus carota,Daucus carota,L.,NaN,NaN,NaN,...,NaN,NaN,Si,NaN,NaN,NaN,NaN,TIRFAA,NaN,Daucus carota
10613,23598.0,E,NaN,10605.0,Thymus mastichina,Thymus mastichina,(L.) L.,NaN,NaN,NaN,...,NaN,NaN,Si,NaN,NaN,NaN,NaN,-,NaN,Thymus mastichina
10614,23599.0,E,NaN,10606.0,Lactuca serriola,Lactuca serriola,L.,NaN,NaN,NaN,...,NaN,NaN,Si,NaN,NaN,NaN,NaN,-,NaN,Lactuca serriola


In [169]:
testdf = testdf[testdf["SOILTYPE"].notna()]
location["soil_type"] = data["SOILTYPE"]

In [186]:
import pandas as pd
from datetime import datetime
import re

def clean_date(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # 1. Match full date: "1960-09-13", possibly with time
    match_full_date = re.match(r"^(\d{4})-(\d{2})-(\d{2})", value)
    if match_full_date:
        try:
            # Validate that it's a real date
            return pd.to_datetime(f"{match_full_date.group(1)}-{match_full_date.group(2)}-{match_full_date.group(3)}").strftime("%Y-%m-%d")
        except:
            pass  # fall through if invalid (e.g., day 00)

    # 2. Match year-month with day 00 → treat as partial: "1960-09-00"
    match_ym00 = re.match(r"^(\d{4})-(\d{2})-00$", value)
    if match_ym00:
        return f"{match_ym00.group(1)}-{match_ym00.group(2)}"

    # 3. Match valid year-month: "1960-09"
    match_ym = re.match(r"^(\d{4})-(\d{2})$", value)
    if match_ym:
        return value

    # 4. Match just a year: "1960", or malformed like "1960----"
    match_y = re.match(r"^(\d{4})(?:-*|----)?$", value)
    if match_y:
        return match_y.group(1)

    # 5. Match cases like "1960-verano", "1960 primavera", etc.
    match_y_text = re.match(r"^(\d{4})[^\d]*.*$", value)
    if match_y_text:
        return match_y_text.group(1)

    return np.nan  # could not interpret


# Aplicar la función a la columna
location["collecting_time"] = location["collecting_time"].apply(clean_date)
location["acquisition_time"] = location["acquisition_time"].apply(clean_date)

# Mostrar resultado
location

,uniqid,accession_number,national_catalogue_code,acquisition_time,collecting_time,soil_type,latitude,longitude,coordinate_uncertainty,geodetic_datum,maximum_elevation,minimum_elevation,terrain_inclination,country_name,country_2letter_code,first_admin_subdivision,second_admin_subdivision,third_admin_subdivision,fourth_admin_subdivision,locality_description
0,20250611160554529107,1.0,NC058636,1960,1960,NaN,3934--N,00239--E,NaN,NaN,15.0,15.0,NaN,ESP,NaN,Baleares,Baleares,Palma de Mallorca,Palma de Mallorca,NaN
1,20250611160554530793,2.0,NC058637,1960,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ESP,NaN,Andalucia,Granada,NaN,"Sierra Nevada,Veleta",NaN
2,20250611160554532136,3.0,NC058638,1960,1960,NaN,3934--N,00239--E,NaN,NaN,15.0,15.0,NaN,ESP,NaN,Baleares,Baleares,Palma de Mallorca,Palma de Mallorca,NaN
3,20250611160554533994,4.0,NC058639,1960,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J.B. Sacavem,NaN
4,20250611160554535192,5.0,NC058640,1960,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ESP,NaN,Andalucia,Granada,NaN,"Sierra Nevada,Veleta",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10611,20250611160610457368,10603.0,NaN,NaN,2021-07-28,NaN,41061302,-3535046,NaN,WGS84,NaN,NaN,NaN,ESP,NaN,"Madrid, Comunidad de",MADRID,Montejo de la Sierra,"Finca privada, Reserva de la Biosfera Sierra d...",NaN
10612,20250611160610458833,10604.0,NaN,NaN,2021-07-29,NaN,41042947,-3566698,NaN,WGS84,NaN,NaN,NaN,ESP,NaN,"Madrid, Comunidad de",MADRID,Madarcos,"Finca privada, Reserva de la Biosfera Sierra d...",NaN
10613,20250611160610460460,10605.0,NaN,NaN,2021-07-29,NaN,410431,-3565898,NaN,WGS84,NaN,NaN,NaN,ESP,NaN,"Madrid, Comunidad de",MADRID,Madarcos,"Finca privada, Reserva de la Biosfera Sierra d...",NaN
10614,20250611160610461989,10606.0,NaN,NaN,2021-08-19,NaN,4106129,-3535056,NaN,WGS84,NaN,NaN,NaN,ESP,NaN,"Madrid, Comunidad de",MADRID,Montejo de la Sierra,"Finca privada, Reserva de la Biosfera Sierra d...",NaN


In [156]:
from datetime import datetime
import time
lista = []
for i in range(1, len(location["uniqid"])+1):
    now = datetime.now()
    now = now.strftime('%Y%m%d%H%M%S%f')
    lista.append(now)
    time.sleep(0.001)
location["uniqid"] = lista
location

,uniqid,accession_number,national_catalogue_code,acquisition_time,collecting_time,soil_type,latitude,longitude,coordinate_uncertainty,geodetic_datum,maximum_elevation,minimum_elevation,terrain_inclination,country_name,country_2letter_code,first_admin_subdivision,second_admin_subdivision,third_admin_subdivision,fourth_admin_subdivision,locality_description
0,20250611160554529107,1.0,NC058636,1960.0,1960----,NaN,3934--N,00239--E,NaN,NaN,15.0,15.0,NaN,ESP,NaN,Baleares,Baleares,Palma de Mallorca,Palma de Mallorca,NaN
1,20250611160554530793,2.0,NC058637,1960.0,1960----,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ESP,NaN,Andalucia,Granada,NaN,"Sierra Nevada,Veleta",NaN
2,20250611160554532136,3.0,NC058638,1960.0,1960----,NaN,3934--N,00239--E,NaN,NaN,15.0,15.0,NaN,ESP,NaN,Baleares,Baleares,Palma de Mallorca,Palma de Mallorca,NaN
3,20250611160554533994,4.0,NC058639,1960.0,1960----,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J.B. Sacavem,NaN
4,20250611160554535192,5.0,NC058640,1960.0,1960----,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ESP,NaN,Andalucia,Granada,NaN,"Sierra Nevada,Veleta",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10611,20250611160610457368,10603.0,NaN,NaN,2021-07-28 00:00:00,NaN,41061302,-3535046,NaN,WGS84,NaN,NaN,NaN,ESP,NaN,"Madrid, Comunidad de",MADRID,Montejo de la Sierra,"Finca privada, Reserva de la Biosfera Sierra d...",NaN
10612,20250611160610458833,10604.0,NaN,NaN,2021-07-29 00:00:00,NaN,41042947,-3566698,NaN,WGS84,NaN,NaN,NaN,ESP,NaN,"Madrid, Comunidad de",MADRID,Madarcos,"Finca privada, Reserva de la Biosfera Sierra d...",NaN
10613,20250611160610460460,10605.0,NaN,NaN,2021-07-29 00:00:00,NaN,410431,-3565898,NaN,WGS84,NaN,NaN,NaN,ESP,NaN,"Madrid, Comunidad de",MADRID,Madarcos,"Finca privada, Reserva de la Biosfera Sierra d...",NaN
10614,20250611160610461989,10606.0,NaN,NaN,2021-08-19 00:00:00,NaN,4106129,-3535056,NaN,WGS84,NaN,NaN,NaN,ESP,NaN,"Madrid, Comunidad de",MADRID,Montejo de la Sierra,"Finca privada, Reserva de la Biosfera Sierra d...",NaN


In [187]:
location.to_csv("location.csv")